# 对于 Meanflow 的改进

本章为你介绍 Improved Meanflow 与 Euler Meanflow，他们极大程度解决了原始 Meanflow 在工程实现上的一些困难。

# Improved Meanflow

推荐你读 https://arxiv.org/abs/2512.02012 Improved Mean Flows: On the Challenges of Fastforward Generative Models 这是 iMF 原文。

首先我们指出，Meanflow 的平均矢量场微分建立在对于神经网络自己的微分上而不是真实的平均矢量场，这从数学法理上并不本质合理，并且这一微分操作还带来了额外的计算开销。更多的，Meanflow 的 CFG 内化训练中，向量外推系数 $\omega $ 是固定的，这意味着模型只可以进行特定向量外推系数的推理。我们在实际训练中观察到固定的 $\omega $ 系数不足以满足多样的现实推理环境。

我们提出 Improved Meanflow 解决这些问题。

## 基本逻辑

我们保持所有来自上一章 Meanflow 的记号。

我们指出一件事，MF 所追求的 $u_{target}$ 是可以被分解为瞬时速度项与微分项的。回顾上一章中的 MF 恒等式
$$\underbrace{u(z_t, r, t)}_{\text{average vel.}} = \underbrace{v(z_t, t)}_{\text{instant. vel.}} - (t - r) \underbrace{\frac{d}{dt} u(z_t, r, t)}_{\text{time derivative}}$$


下面这张图展示了这个事实。橙色部分是真实瞬时速度，蓝色部分是平均矢量场方向，灰色则是微分项。

<img src="./assets/iMF.png" width="600" height="300">

然而，我们极难通过采样估计边缘矢量场的微分项 $- (t - r) {\frac{d}{dt} u(z_t, r, t)}$。因此 MF 提出估计神经网络所学习到的边缘矢量场 $u_\theta$ 的微分来替代，也就是
$$u_{\text{target}} = v(z_t, t) - (t - r) \left( v(z_t, t) \partial_z u_\theta + \partial_t u_\theta \right)$$
但是这就带来了模型目标函数偏移的问题。学习一个动态的目标是困难的。

原始 MF 提出拟合 $u_{target}$ 的想法，即损失函数
$$\mathcal{L}_\theta = \mathbb{E} \| u_\theta(z_t,r,t) - \text{sg}(u_{target})\|_2^2 $$

现在我们开始提出神奇的改进，我们对 MF 恒等式做移项
$${v(z_t, t)} = {u(z_t, r, t)} + (t - r) {\frac{d}{dt} u(z_t, r, t)}$$
定义新的学习对象 $V_\theta$
$$ V_\theta = {u_\theta (z_t, r, t)} + (t - r) {\frac{d}{dt} u_\theta (z_t, r, t)}$$
转化原始损失函数
$$\mathcal{L}_\theta = \mathbb{E} \| V_\theta  - v(z_t, t) \|_2^2 $$
完全没有学习目标偏移问题了。我们将所有跟随神经网络的项视为我们本来就应该优化的对象。

这看起来非常像玩一个文字游戏，实际上不是这样。根本原因在于原始 MF 中 JVP 项的计算是依赖真实速度 $v(z_t,t)$ 的，然而我们现在将 JVPA 移动到左边作为一个学习对象，就意味着我们对于 JVP 的计算绝对不能依赖原始真实速度。下面我详细解释。

我们讨论一下 $V_\theta$ 计算问题。首先展开
$$V_\theta = {u_\theta (z_t, r, t)} + (t - r) \left( v(z_t, t) \partial_z u_\theta + \partial_t u_\theta \right)$$
原始 MF $V_\theta$ 还需要真实瞬时矢量场 $v(z_t, t)$ 才能计算。在原始 MF 中，由于微分项仅仅在训练时出现，这个问题没有被暴露。然而现在我们将其视为模型学习对象一部分，在推理时我们无法满足给出真实矢量场这个要求。

于是改进就是将这个矢量场改为模型的矢量场 $v_\theta$
$$V_\theta = {u_\theta (z_t, r, t)} + (t - r) \left( v_\theta(z_t, t) \partial_z u_\theta + \partial_t u_\theta \right)$$
这是自然的，因为 $v_\theta(z_t, t) = u_\theta(z_t, t, t)$ 确保了我们可以通过模型获取真实瞬时矢量场。

将真实矢量场改为模型预测矢量场是一个关键的改动，这就是我们上述说的取消 JVP 项对于真实矢量场的依赖，进而将其视为优化对象一部分。

现在可以构建训练流程。但是我们最后提醒一件事，那就是我们在计算 $V_\theta$ 时，我们还是会像原始 Meanflow 那样为微分部分加上停训算子 $\text{sg}$。原因是如果不这么做，对于微分的优化过程是一个更高阶的梯度传播过程，这种混合二阶偏导数的计算会造成巨大的显存和算力负担。

我们就不详细赘述训练过程了，原因是极其简单且类似 Meanflow 原始训练过程。我们指出，其实变化仅仅是矢量场来源从真实矢量场变成模型拟合的矢量场，保证推理时的合法性。

下面是完整的训练流程。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{improved MeanFlow: training.} \\
\text{Note: in PyTorch and JAX, } \texttt{jvp} \text{ returns the function output and JVP.} \\
\hline
\\
\color{teal}{\#\ fn(z, r, t)\text{: function to predict } u} \\
\color{teal}{\#\ x\text{: training batch}} \\
\\
t, r = \text{sample\_t\_r}() \\
e = \text{randn\_like}(x) \\
\\
z = (1 - t) \ast x + t \ast e \\
\\
\color{teal}{\#\ \text{instantaneous velocity } v \text{ at time } t} \\
v = \text{fn}(z, t, t) \\
\\
\color{teal}{\#\ \text{predict } u \text{ and dudt}} \\
u, \text{dudt} = \texttt{jvp}(\text{fn}, (z, r, t), (v, 0, 1)) \\
\\
\color{teal}{\#\ \text{compound function } V} \\
V = u + (t - r) \ast \text{stopgrad}(\text{dudt}) \\
\text{error} = V - (e - x) \\
\\
\text{loss} = \text{metric}(\text{error}) \\
\\
\hline
\end{array}$$

由于 iMF 推理几乎与原始 MF 一致，我们同样不赘述。

## 工程技巧

我们来讲述一些工程技巧，主要是对于某些细微的参数嵌入方式或者采样方式做优化。

### CFG 内化

原始的 Meanflow 将 CFG 在训练中内化，使得实际中可以一步生成而无需为了向量外推调用两次模型。不过一个问题就是训练中的外推系数 $\omega$ 是固定的。我指 
$$v^{\text{cfg}}(z_t, t \mid \mathbf{c}) \triangleq \omega v(z_t, t \mid \mathbf{c}) + (1 - \omega) v(z_t, t)$$
假如我们在训练中设置 $\omega$ 为 $\omega_0$，实际推理中却想要 $\omega_1$ 的效果，我们必须重新训练模型。这对于实际工业生产非常不友好。

我们指出，CFG 引导系数 $\omega$ 最好随着推理环境与模型训练程度不同而变化。下面这张图展示了 CFG 系数随着训练轮次深入与推理步数变化而产生变化的过程。可以看到，随着训练深入或随着推理步数增加，CFG 系数的最优解在逐渐向低偏移。单一的 CFG 系数显然无法做到时刻最优。 

<img src="./assets/CFG.png" width="500" height="240">

解决方法很简单，我们给模型输入 $\omega$ 参数即可。我们重新定义 
$$V_\theta(\cdot \mid \mathbf{c}, \omega) \triangleq u_\theta(z_t \mid \mathbf{c}, \omega) + (t - r)\frac{d}{dt}u_\theta(z_t \mid \mathbf{c}, \omega)$$
其他部分则同样加上对于 $\omega$ 的条件。 

更多的，训练中关于 $\omega $ 的采样方式也可以衍生出多种优化方法。比如原文提出，对于某个时间区间 $[t_{min}, t_{max}]$ 之外的时间步，CFG 训练全部会被关闭，而在这个区间上则采取独特的 $\omega$ 采样密度函数。

### 条件嵌入

正如上述所说，我们的模型接收诸多参数 
$$u_\theta = u_\theta(z_t \mid r, t, c, \Omega )$$
其中 $z_t$ 是当前加噪图像，$r,t$ 是平均时间的始终点，$c$ 是类别标签，$\Omega$ 是对于 CFG 引导系数的条件。

一般而言，这些条件嵌入方式是我们在 DiT 中提到的 adaLN-Zero 架构，但是 iMF 提出了 In-context Conditioning 会更好。这其实很反直觉，因为 In-context Conditioning 实际上就是简单将条件嵌入编码之后直接与图像编码拼接在一起。但是这是合理的，因为原始 adaLN-Zero 的做法大幅压缩了信息量，对于简单条件嵌入效果极佳，面对复杂条件嵌入就表现得力不从心。

简单来说，原始图像经过编码之后是 $(Batch, Seq, D)$，我们将每个条件也编码为 $(Batch, Num, D)$ 的形状，然后直接将所有张量沿着序列维度拼接，视为统一张量输入网络处理，Transformer 会自己处理各个条件之间语义联系。其中 $Num$ 被推荐设置为 $4$ 或 $8$ 以平衡信息量与计算成本。 

下面这张图展示了这个架构。黄色部分是主要的图像编码，绿色粉色蓝色则是各类条件的嵌入，我们将其拼接交给网络处理。

<img src="./assets/iMF_cond.png" width="360" height="280">

In-context Conditioning 的好处是削减了原本 adaLN-Zero 的条件嵌入网络体积。原作者最后发现可以直接将模型参数量减少为原本 $\frac{2}{3}$。这个红利非常大。

## iMF 的总结

iMF 是何恺明团队对于 MF 成果继续跟进的结果，他们解决了一些原始 MF 数学法理上的痛点，如将回归目标改进到 Ground-Truth，进而成为标准的回归任务。

然而 MF 实际计算中的大敌，也就是 JVP 微分仍没有被找到合适的办法取缔或者近似。下面为你介绍 Euler Meanflow。他们找到了这一问题的答案。

# Euler Meanflow

推荐你读 https://arxiv.org/abs/2602.02571 Trajectory Consistency for One-Step Generation on Euler Mean Flows 这是 Euler Meanflow 原文。

## 基本逻辑

我们更换一下记号，并且保持所有来自 Flow Matching 那一章的记号。简记我们探讨的数据空间为 $V$，无论是潜空间还是像素空间。

回顾一下。全局边缘概率分布需要与边缘矢量场满足连续性方程
$$\frac{\partial p_t(x)}{\partial t} + \nabla \cdot (p_t(x) u_t(x)) = 0$$
并且轨迹与边缘矢量场存在关系 
$$ x_0 \sim p_{\text{init}}, \quad \frac{\text{d}}{\text{dt}} x_t = u_t(x_t)$$
其中 $$x_t \sim p_t \quad (0 \leq t \leq 1)$$
我们利用关系 $\phi_t(x_0) = x_t$，得到映射作为泛函的关系
$$\frac{\partial}{\partial t} \phi_t = u_t \circ \phi_t, \quad \phi_0 = \text{Id}_V$$

更多的，边缘概率密度函数与流还存在直接的关系，也就是测度的 Push-forward
$$p_t = [\phi_t]_* p_0$$

现在我们开始引入一些新的表述。我们定义流映射
$$\phi_{t \rightarrow r} = \phi_r \circ \phi_t^{-1}$$
这很好理解，简而言之就是 $\phi_{t \rightarrow r}(x_t) = x_r$。

更多的，这个映射必须满足两个边界条件。首先，$\phi_{t \rightarrow t}(x) = x$，这是因为可逆映射复合其逆映射就是恒等映射。其次的，我们有$$\lim_{r \to t} \frac{\partial}{\partial r} \phi_{t \to r}(x) = \frac{\partial}{\partial r} \phi_{t \to r}(x) \bigg|_{r=t} = u_t\left( \phi_{t \to t}(x) \right) = u_t(x)$$
这也同样易于理解，粒子在时间步 $t$ 刚刚离开 $x$ 的速度应该就是 $u_t(x)$。

我们指出，以上这两个边界条件是易于模型学习的，因为我们只需要监督 $r=t$ 情况下流映射输出即可。传统的一致性模型或是 Meanflow 最大困难就是监督长程流映射的一致性。我们详谈。

考虑这样一条轨迹 $(x_t)_{t \in [0,1]}$ 并且 $x_t = \phi_t (x_0)$。对于任意的 $t \le s \le r \in [0,1]$，我们有
$$\phi_{t \rightarrow r}(x_t) = \phi_{s \rightarrow r}(x_s), \quad x_s = \phi_{t \rightarrow s}(x_t)$$
更多的，赋予极限 $\lim {s \to t}$，可以得到微分形式
$$\frac{d}{ds} \Big[ \phi_{t \to r}(x) \Big] = \frac{d}{ds} \Big[ \phi_{s \to r}\left( \phi_{t \to s}(x) \right) \Big]$$
左式是 $0$，右式是个多元复合函数微分，展开得到
$$0 = \partial_t \phi_{s \to r}\left( \phi_{t \to s}(x) \right) + \partial_x \phi_{s \to r}\left( \phi_{t \to s}(x) \right) \cdot \partial_s \phi_{t \to s}(x)$$
最后令中间时间 $s \to t$ 逼近起点
$$0 = \partial_t \phi_{t \to r}(x) + \partial_x \phi_{t \to r}(x) \left. \left( \partial_s \phi_{t \to s}(x) \right) \right|_{s=t}$$

现在我们可以给出自然的一致性训练的目标函数
$$\mathcal{L}^C(\theta) = \mathbb{E}_{t, s, r, x_t = (1-t)x_0 + tx_1, x_1 \sim p_{\text{data}}, x_0 \sim p_0} \frac{1}{w(t, r)} \left\| \phi^{\theta}_{t \to r}(x_t) - \phi^{\theta}_{s \to r} \left( \phi^{\theta}_{t \to s}(x) \right) \right\|_2^2$$
此处 $\frac{1}{w(t, r)}$ 是可调节权重。

然而，作者指出一件事，尽管上述的目标函数高度自洽，但是其拥有无数个平凡解。这是因为这一目标函数没有任何来自真实数据分布的监督。所以我们必须引入对于真实数据的学习。在我们先前介绍的 Consistency Models 中，我们通过将 $\phi_{t \to s}$ 这一映射结果交给被蒸馏的 FM 模型给出，以此引入了模型对于真实数据的学习。

所以，如何学习这个流映射？在 Euler Meanflow 之前，主要分为两大类别。首先是我们上一章介绍的 Meanflow 原版，我们将边缘矢量场拆分为真实瞬时速度与边缘矢量场微分，并且让模型学习边缘矢量场分布，好处是提供了强大的直接监督学习，坏处则是 JVP 求偏微分带来巨大的计算负担。另一个派别则是 Split-Mean Flow 与 Short-Cut Models，后者我们会在后续详细介绍，主要方法是渐进延展，逐步产生长程流映射，坏处则是误差会逐渐累积至无法接受。

现在我们开始讲述 Euler Meanflow 的做法。

定义类似 Meanflow 平均矢量场的内容
$$ u_{t \to r}(x) = \frac{\phi_{t \to r}(x) - x}{r - t}$$
换句话说
$$u_{t \to r}(x_t) = \frac{\phi_{t \to r}(x_t) - x_t}{r - t} = \frac{x_r - x_t}{r - t} = \frac{1}{r -t}\int_t^r u_{\tau \to \tau}(x_\tau) d\tau$$
这就是 Meanflow 所定义的内容。

同样的，我们拥有天生的一致性条件
$$(r-t) u_{t \to r}(x_t) = (s-t) u_{t \to s}(x_t) + (r-s) u_{s \to r}(x_s)$$
两边同除 $s -t $
$$u_{t \to s}(x_t)  = (r-s)\frac{u_{t \to r}(x_t) - u_{s \to r}(x_s)}{(s -t)} +  u_{t \to r}(x_t)$$
我们设置 $ s = t + \Delta t$，其中 $\Delta t$ 是个固定极小步长。

这里我们做一个近似，这里利用到微分的定义
$$\phi_{t \to s}(x) \approx  \phi_{t \to t}(x) + \left. \frac{\partial \phi_{t \to s}(x)}{\partial s} \right|_{s=t} (s - t)$$

现在代入 $s = t + \Delta t$
$$u_{t \to t + \Delta t}(x_t)  = (r-t - \Delta t)\frac{u_{t \to r}(x_t) - u_{t + \Delta t \to r}(x_{t + \Delta t})}{\Delta t} +  u_{t \to r}(x_t)$$
做近似 $u_{t \to t + \Delta t}(x_t) \approx  u_{t \to t}(x_t)$
$$ u_{t \to r}(x_t) = u_{t \to t }(x_t) + (r-t - \Delta t)\frac{ u_{t + \Delta t \to r}(x_{t + \Delta t})- u_{t \to r}(x_t)}{\Delta t}$$
再做近似 $x_{t + \Delta t} \approx \Delta t u_{t \to t +\Delta t}(x_t) + x_t \approx \Delta t u_{t \to t}(x_t) + x_t$，最终我们得到工程上的平均矢量场获得方式
$$ u_{t \to r}(x_t) = u_{t \to t }(x_t) + (r-t - \Delta t)\frac{ u_{t + \Delta t \to r}(\Delta t u_{t \to t}(x_t) + x_t)- u_{t \to r}(x_t)}{\Delta t}$$

现在我们可以定义损失函数
$$\mathcal{L}^E(\theta) = \mathbb{E}_{t, r, x_1 \sim p_1, x \sim p_t(x \mid x_1), x' = \text{sg}\left(\Delta t u^{\theta}_{t \to t}(x)\right) + x} \left\| u^{\theta}_{t \to r}(x) - \left( u_t(x \mid x_1) + (r - t - \Delta t)_+ \cdot \text{sg}\left( \frac{u^{\theta}_{t + \Delta t \to r}(x') - u^{\theta}_{t \to r}(x)}{\Delta t} \right) \right) \right\|^2$$
以上损失函数中，当 $r =t$，该损失会直接退化为原始 Flow Matching，这意味着模型正在学习原始的真实瞬时矢量场。

关于优化 $\mathcal{L}^E(\theta)$ 等价优化 $\mathcal{L}^C(\theta)$ 这件事，也就是近似后损失函数所保留的合法性，我们不做详细证明。一件符合直觉的事情是，既然我们对所有部分采取了合理的估计并且控制了变量的变化范围，估计后函数等价原始函数应该是理所应当的。更多的，作者给出实验数据表明最终估计函数 $\mathcal{L}^E(\theta)$ 与 $\mathcal{L}^C(\theta)$ 差距保持在原始 FM 损失函数量级。换言之，只要我们保证模型良好学习了每个点的瞬时矢量场，我们就可以保证整体良好的学习。

这里我们再讨论一件事。一个有趣的问题是，Euler Meanflow 是否采纳了 Improved Meanflow 的优化结果？iMF 将所有关于神经网络自己的产生的预测项视为完整优化对象 $V_\theta$，留下标准的 Ground-Truth 回归目标 $u_t(x \mid x_1)$。但是 Euler Meanflow 似乎没有做这件事。

这个问题的答案是，是也不是，或者两者某种意义上是等价的。首先一个事实是，Euler Meanflow 的成果仅仅比 iMF 晚两个月发表，他们甚至没有做自己与 iMF 的性能对比。其次的，假如我们效仿 iMF 将所有关于神经网络自己的产生的预测项加在一起视为优化的整体，实际上和目前的损失函数没有任何差别。原因就是我们对于微分近似项的加上的停训算子 $\text{sg}$，这导致优化器完全无视了这一项的优化趋向，无论我们如何改变这一项计算方式都不会产生原理性的变化。

所以这个答案听起来很显然。iMF 的路线是保留 JVP 计算优化数学原理合法性，Euler MF 则选择直接近似替代 JVP 项。

但是一个事实是，Euler MF 对于单次数据点的训练需要调用三次神经网络，iMF 却仅仅需要两次调用。我指的是 iMF 需要计算 $u_\theta (z_t, r, t)$ 与 $ v_\theta(z_t, t) = u_\theta (z_t, t, t)$，而 Euler MF 则需要计算 $u_{t \to r}^\theta(x)$ 与 $u^\theta_{t \to t}(x)$ (为了计算 $x'$) 以及最终 $u_{t + \Delta t \to r}^\theta(x')$。不过这似乎是值得的，因为 Euler MF 作者做出一个重大改动就是添加一个对于 $u^\theta_{t \to t}(x)$ 的直接预测头，这导致对于这个新的神经网络，实际上我们只需要调用两次。另外 iMF 还需要调用 JVP 计算偏微分。总之，Euler MF 作者最终的数据显示他们在显存与算力用量上有巨大优化。

下面这张图展示了作者给出的独特神经网络架构。可以看到这个双头结构帮助减少了总的网络调用次数。不过争议的一点是，尽管原文中强调对于$u^\theta_{t \to t}(x)$ 的辅助预测头是轻量的，这样仍应会增加总的参数量。

<img src="./assets/EMF.png" width="340" height="400">

现在理论上已经可以开始训练了，我们想说最后一个优化。作者受到何恺明团队 Just-Image Transformer 的思想影响，决定做一次等价的代换。 

## $x_1$- Prediction Euler MF

作者在真实训练中发现，$u_t^\theta$ 难以拟合真实瞬时速度场 $u_t$。而 JiT 的成果近期指出，直接预测图像本身而不是间接的矢量场或噪声反而是更直接迅速的学习方式。于是作者提出一种代换，我们称之为 Endpoint 预测。这种预测方法最早来自于 Functional Meanflow，非常有趣。

定义新的平均场
$$\tilde{x}_{t \to r}(x) = (1 - t) \frac{\phi_{t \to r}(x) - x}{r - t} + x$$
我先解释为什么这样定义新的场是自然的。首先新的场与原平均矢量场有关系
$$\tilde{x}_{t \to r}(x) = (1 - t) u_{t \to r}(x) + x$$
假设我们的粒子在时间步 $t$ 的道中位置是 $x$，并且拥有速度 $u_{t \to r}(x)$，那么如果粒子从当前时间步 $t$ 一直运动到时间尽头 $1$，它就会达到点 $\tilde{x}_{t \to r}(x)$。

更多的，假如我们取 $r=t$，那么可以得到
$$\tilde{x}_{t \to t}(x) = (1 - t) u_{t \to t}(x) + x$$
如果我们使用最优路径训练模型，实际上条件路径是直线且瞬时速度 $u_{t \to t}(x \mid x_1)$ 严格指向终点，那么 $\tilde{x}_{t \to t}(x \mid x_1)$ 会成为终点的真实图像 $x_1$。

最重要的一条是，当我们给定初始点 $x$，$u_{t \to r}$ 与 $\tilde{x}_{t \to r}(x)$ 存在一一对应关系。

我需要说明的是，这一 $x_1$-prediction 的场形式应该最早来自何恺明的 Just-image Transformer。他们发现虽然预测原始图像与预测噪声数学上等价，但是真实的数据流形是低维空间上的，而噪声预测反而将问题扩大到不必要的高维空间。

我们继续对于损失函数的代换。我们可以利用原平均矢量场的一致性关系，代换得到
$$\tilde{x}_{t \to r}(x_t) = \tilde{x}_{t \to s}(x_t) + (r - s) \frac{(1 - t)}{(1 - r)} \frac{\tilde{x}_{s \to r}(x_s) - \tilde{x}_{t \to r}(x_t)}{s - t}$$

假如我们直接令 $s \to t$，可以得到
$$\tilde{x}_{t \to r}(x_t) = \tilde{x}_{t \to t}(x_t) + \frac{(1 - t)}{(1 - r)} \lim_{s \to t} \Big [ (r - s) \frac{\tilde{x}_{s \to r}(x_s) - \tilde{x}_{t \to r}(x_t)}{s - t} \Big ]$$

$$ = \tilde{x}_{t\to t}(x_t)+\frac{(1-t)(r-t)}{1-r}\left.\frac{d}{ds}\tilde{x}_{s\to r}(x_s)\right|_{s=t}$$

更多的，我们保持 $s = t + \Delta t$ 的设置，并且做近似 $ \tilde{x}_{t \to s}(x_t) \approx \tilde{x}_{t \to t}(x_t)$

$$\tilde{x}_{t \to r}(x_t) \approx {\tilde{x}_{t \to t}}(x_t) + (r - t - \Delta t) \frac{(1 - t)}{(1 - r)} \frac{\tilde{x}_{t + \Delta t \to r}(x_{t + \Delta t}) - \tilde{x}_{t \to r}(x_t)}{\Delta t} $$
此处
$$x_{t + \Delta t} = \frac{\Delta t}{1 - t} (\tilde{x}_{t \to s}(x_t) - x_t) + x_t \approx \frac{\Delta t}{1 - t} ({\tilde{x}_{t \to t}}(x_t) - x_t) + x_t$$

同样的，我们将 $\tilde{x}_{t \to t}(x_t)$ 代换为条件场 $\tilde{x}(x \mid x_1 ) = x_1 $
$$\mathcal{L}^{E'}(\theta) = \mathbb{E}_{t, r, x_1 \sim p_1, x \sim p_t(x \mid x_1), x' = \text{sg} \left( \Delta t \frac{\tilde{x}^{\theta}_{t \to t}(x) - x}{1 - t} \right) + x} \\

\left[ \left\| \tilde{x}^{\theta}_{t \to r}(x) - \left( \tilde{x}_{t \to t}(x \mid x_1) + (r - t - \Delta t)_+ \frac{1 - t}{1 - r} \cdot \text{sg} \left( \frac{\tilde{x}^{\theta}_{t + \Delta t \to r}(x') - \tilde{x}^{\theta}_{t \to r}(x)}{\Delta t} \right) \right) \right\|^2 \right]$$
同样的，当 $r=t$，这一损失函数退化为 FM 原始损失函数。

最后一个工程细节是，当 $r=t$，作者给出一个权重上的优化，即为损失函数附上关于时间步 $t$ 权重
$$\frac{1}{(1-t)^2}$$
不过我们会阻止除以零的错误，$1-t$ 最低为 $0.02$。

## 训练

我们快速说一说具体的训练。假设我们有真实数据集 $\mathcal{D}$，噪声采样分布 $\mathcal{N}$，时间步采样分布 $\mathcal{T}$ 以及更多超参数。

其中，时间步采样分布 $\mathcal{T}$ 是值得关注的。作者设置了默认的 $\mathcal{U}[0,1]$ 与 Log-Norm 采样，同时设置一个概率 $\alpha$ 表示直接采样 $r = t$。这保证了对于瞬时速度与平均速度的拟合训练都能够良好进行。

同时他们也采用了我们在 MF 中提到的自适应损失权重稳定梯度。

首先采样真实数据点 $x_1$，条件标签 $C$，初始噪声 $x_0$ 与时间步 $t, r$。

计算前向加噪 $x_t = (1-t)x_0 + t x_1$。

计算 CFG 增强训练的 $u_t(x \mid x_1)$。将时间步 $t, r$ 与带噪图像 $x_t$ 与条件 $C $ 输入模型得到 $u^\theta_{t \to r}(x_t,C)$。再计算 $u^\theta_{t \to t}(x_t, C)$ 以计算 $x_{t + \Delta t}$，最后计算 $u^\theta_{t + \Delta t \to r}(x_{t + \Delta t},C)$。

现在可以计算最终损失函数。遍历一个 Batch 之后反向传播更新参数。

完整训练算法如下。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Euler Mean Flow: Training} \\
\textit{Highlighted steps are used for conditional generation. } C \text{ repre-} \\
\textit{sents the class label, and } C_0 \text{ the corresponding unconditional} \\
\textit{label. } w \text{ and } k \text{ are parameter for CFG} \\
\hline
\textbf{Require: } \text{Dataset } \mathcal{D}, \text{parameters } \theta, \text{learning rate } \eta, \text{noise} \\
\qquad\ \text{sampler } \mathcal{N}, \text{time sampler } \mathcal{T} \\
\begin{aligned}
1: & \quad \textbf{repeat} \\
2: & \qquad \text{Sample } x_1, \color{blue}{C} \color{black}{\ \sim \mathcal{D}, x_0 \sim \mathcal{N}, t, r \sim \mathcal{T}} \\
3: & \qquad x_t \leftarrow (1 - t)x_0 + t x_1 \\
4: & \qquad \textbf{if } \text{u-prediction} \textbf{ then} \\
5: & \qquad\quad \color{blue}{U^u \leftarrow u_{t \to t}^{\theta}(x_t, C_0), U^c \leftarrow u_{t \to t}^{\theta}(x_t, C)} \\
6: & \qquad\quad u_t(x \mid x_1) \leftarrow \color{blue}{(1 - w - k)U^u + kU^c + w}\color{black}{(x_1 - x_0)} \\
7: & \qquad\quad x_{t+\Delta t} \leftarrow \Delta t U_t^c + x_t \\
8: & \qquad\quad \mathcal{L} \leftarrow \left\| u_{t \to r}^{\theta}(x_t, \color{blue}{C}\color{black}{)} - \text{sg}\left( u_t(x \mid x_1) + (r - t - \right.\right. \\
   & \qquad\quad \left.\left. \Delta t)_+ \frac{u_{t+\Delta t \to r}^{\theta}(x_{t+\Delta t}, \color{blue}{C}\color{black}{)} - u_{t \to r}^{\theta}(x_t, \color{blue}{C}\color{black}{)}}{\Delta t} \right) \right\|^2 \\
   & \qquad \textbf{else if } x_1\text{-prediction} \textbf{ then} \\
9: & \qquad\quad \color{blue}{X^u \leftarrow \tilde{x}_{t \to t}^{\theta}(x_t, C_0), X^c \leftarrow \tilde{x}_{t \to t}^{\theta}(x_t, C)} \\
10:& \qquad\quad \tilde{x}_t(x \mid x_1) \leftarrow \color{blue}{(1 - w - k)X^u + kX^c + w}\color{black}{x_1} \\
11:& \qquad\quad x_{t+\Delta t} \leftarrow \Delta t \frac{X^c - x_t}{1 - t} + x_t \\
12:& \qquad\quad \mathcal{L} \leftarrow \left\| \tilde{x}_{t \to r}^{\theta}(x_t, \color{blue}{C}\color{black}{)} - \text{sg}\left( \tilde{x}_t(x \mid x_1) + (r - t - \right.\right. \\
   & \qquad\quad \left.\left. \Delta t) _+ \frac{1-t}{1-r} \frac{\tilde{x}_{t+\Delta t \to r}^{\theta}(x_{t+\Delta t},\color{blue}{C}\color{black}) - \tilde{x}_{t \to r}^{\theta}(x_t),\color{blue}{C}}{\Delta t} \right) \right\|^2 \\
13:& \qquad \textbf{end if} \\
14:& \qquad \theta \leftarrow \theta - \eta \nabla_{\theta}\mathcal{L} \\
15:& \quad \textbf{until } \text{convergence}
\end{aligned} \\
\hline
\end{array}$$

## 推理

Euler Meanflow 的采样同样很直接。我们直接考虑从时间步 $t=0$ 到 $1$ 的平均矢量场或是 $x_1$ 预测结果即可。

完整采样算法如下。

$$\begin{array}{l}
\hline
\textbf{Algorithm 2 } \text{Euler Mean Flow: Sampling} \\
\textit{Highlighted parts are used for conditional generation.} \\
\hline
\textbf{Require: } \text{parameters } \theta, \text{learning rate } \eta, \text{noise sampler } \mathcal{N} \\
\begin{aligned}
1: & \quad \textbf{repeat} \\
2: & \qquad \text{Sample } x_0 \sim \mathcal{N} \\
3: & \qquad \textbf{if } u\text{-prediction} \textbf{ then} \\
4: & \qquad\quad x_1 = u_{0 \to 1}^{\theta}(x, \color{blue}{C}\color{black}{)} + x \\
   & \qquad \textbf{else if } x_1\text{-prediction} \textbf{ then} \\
5: & \qquad\quad x_1 = \tilde{x}_{0 \to 1}^{\theta}(x, \color{blue}{C}\color{black}{)} \\
6: & \qquad \textbf{end if} \\
7: & \quad \textbf{until } \text{convergence}
\end{aligned} \\
\hline
\end{array}$$

# 总结

本章介绍了对于原始 Meanflow 方法的两种改进，其中 Improved MF 解决了原始 MF 回归目标偏移的问题，并且提出了更具数学合法性的方案；Euler MF 则提出一种免去 JVP 计算的近似策略，大大减少了计算显存与算力压力。

关于加速生成中的一步生成，我们还远远没有说完。下一章为你介绍 Shortcut Models，另一种一步生成模型范式。